In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import expm_multiply

In [ ]:
# ==============================================================================
# 1. Grid and Domain Parameters
# ==============================================================================
L = 1.0              # Length of the rod
Nx = 51              # Number of spatial grid points
alpha = 0.01         # Thermal diffusivity
t_final = 2.0        # Final simulation time

dx = L / (Nx - 1)
x = np.linspace(0, L, Nx)

# For explicit stability: r = alpha * dt / dx^2 <= 0.5
r_stable = 0.4
dt = r_stable * (dx**2) / alpha
Nt = int(t_final / dt)
t_final = Nt * dt  # Align t_final exactly with time steps

print(f"Grid spacing (dx): {dx:.4f}")
print(f"Time step (dt):    {dt:.6f}")
print(f"Number of steps:   {Nt}")
print(f"Stability parameter (r): {alpha * dt / dx**2:.4f}")

In [ ]:
# ==============================================================================
# 2. Initial and Boundary Conditions (Defined Once)
# ==============================================================================
def get_initial_condition(x_grid):
    """
    Defines the initial temperature distribution at t = 0.
    Currently configured as a Gaussian pulse centered at L/2.
    """
    return np.exp(-100 * (x_grid - 0.5)**2)

def apply_boundary_conditions(u_array, t):
    """
    Applies Dirichlet boundary conditions at the endpoints. 
    Returns a copied array to prevent mutating the input profile in-place.
    """
    u_bc = u_array.copy()  # Ensure independence
    u_bc[0] = 0.0          # Left boundary: u(0, t) = 0
    u_bc[-1] = 0.0         # Right boundary: u(L, t) = 0
    return u_bc

# Generate the master initial temperature profile
u0_master = apply_boundary_conditions(get_initial_condition(x), 0.0)

# Visualize the shared initial condition
plt.figure(figsize=(6, 3))
plt.plot(x, u0_master, 'r-', label='Initial Condition (t=0)')
plt.xlabel('x')
plt.ylabel('Temperature')
plt.title('Shared Initial Temperature Profile')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ==============================================================================
# 3. FTCS (Forward Time, Central Space) - Explicit Scheme
# ==============================================================================
def solve_ftcs(x_grid, u0, alpha, dt, dx, Nt):
    """
    Solves the 1D heat equation using the FTCS explicit scheme.
    """
    u = u0.copy() 
    r = alpha * dt / dx**2
    snapshots = [u.copy()]
    times = [0.0]
    
    for n in range(Nt):
        u_new = u.copy()
        u_new[1:-1] = u[1:-1] + r * (u[2:] - 2*u[1:-1] + u[:-2])
        u_new = apply_boundary_conditions(u_new, (n + 1) * dt)
        u = u_new
        
        save_interval = max(1, Nt // 10)
        if (n + 1) % save_interval == 0 or n == Nt - 1:
            snapshots.append(u.copy())
            times.append((n + 1) * dt)
            
    return snapshots, times

In [ ]:
# ==============================================================================
# 4. BTCS (Backward Time, Central Space) - Implicit Scheme
# ==============================================================================
def solve_btcs(x_grid, u0, alpha, dt, dx, Nt):
    """
    Solves the 1D heat equation using the BTCS implicit scheme.
    """
    u = u0.copy()
    Nx = len(x_grid)
    r = alpha * dt / dx**2
    snapshots = [u.copy()]
    times = [0.0]
    
    A = np.zeros((Nx, Nx))
    for i in range(1, Nx - 1):
        A[i, i-1] = -r
        A[i, i] = 1 + 2*r
        A[i, i+1] = -r
    A[0, 0] = 1.0
    A[-1, -1] = 1.0
    
    for n in range(Nt):
        b = u.copy()
        b = apply_boundary_conditions(b, (n + 1) * dt)
        u_new = np.linalg.solve(A, b)
        u = u_new
        
        save_interval = max(1, Nt // 10)
        if (n + 1) % save_interval == 0 or n == Nt - 1:
            snapshots.append(u.copy())
            times.append((n + 1) * dt)
            
    return snapshots, times

In [ ]:
# ==============================================================================
# 5. Crank-Nicolson - Semi-Implicit Scheme
# ==============================================================================
def solve_crank_nicolson(x_grid, u0, alpha, dt, dx, Nt):
    """
    Solves the 1D heat equation using the Crank-Nicolson semi-implicit scheme.
    """
    u = u0.copy()
    Nx = len(x_grid)
    r = alpha * dt / dx**2
    snapshots = [u.copy()]
    times = [0.0]
    
    A = np.zeros((Nx, Nx))
    B = np.zeros((Nx, Nx))
    for i in range(1, Nx - 1):
        A[i, i-1] = -r / 2.0
        A[i, i] = 1.0 + r
        A[i, i+1] = -r / 2.0
        
        B[i, i-1] = r / 2.0
        B[i, i] = 1.0 - r
        B[i, i+1] = r / 2.0
        
    A[0, 0] = 1.0
    A[-1, -1] = 1.0
    B[0, 0] = 1.0
    B[-1, -1] = 1.0
    
    for n in range(Nt):
        b = B.dot(u)
        b = apply_boundary_conditions(b, (n + 1) * dt)
        u_new = np.linalg.solve(A, b)
        u = u_new
        
        save_interval = max(1, Nt // 10)
        if (n + 1) % save_interval == 0 or n == Nt - 1:
            snapshots.append(u.copy())
            times.append((n + 1) * dt)
            
    return snapshots, times

In [ ]:
# ==============================================================================
# 6. Matrix Exponential - expm_multiply Approach
# ==============================================================================
def solve_matrix_exponential(x_grid, u0, alpha, dt, dx, Nt):
    """
    Solves the 1D heat equation using the matrix exponential (expm_multiply) approach.
    Since boundary conditions are homogeneous Dirichlet (0), we solve the ODE system:
        d(u_interior)/dt = M * u_interior.
    """
    u = u0.copy()
    Nx = len(x_grid)
    snapshots = [u.copy()]
    times = [0.0]
    
    # Evolve the interior grid points
    u_int = u[1:-1]
    n_int = Nx - 2
    
    # Construct the tridiagonal matrix M for the interior points
    coeff = alpha / dx**2
    diagonals = [
        coeff * np.ones(n_int - 1),   # sub-diagonal
        -2 * coeff * np.ones(n_int),  # main diagonal
        coeff * np.ones(n_int - 1)    # super-diagonal
    ]
    M = diags(diagonals, [-1, 0, 1], format='csr')
    
    save_interval = max(1, Nt // 10)
    for n in range(Nt):
        # Evolve interior points by dt using expm_multiply
        u_int = expm_multiply(dt * M, u_int)
        
        # Reconstruct full solution with boundaries
        u_new = np.zeros(Nx)
        u_new[1:-1] = u_int
        u_new = apply_boundary_conditions(u_new, (n + 1) * dt)
        u = u_new
        
        if (n + 1) % save_interval == 0 or n == Nt - 1:
            snapshots.append(u.copy())
            times.append((n + 1) * dt)
            
    return snapshots, times

In [ ]:
# ==============================================================================
# 7. Execution and Verification of Independence
# ==============================================================================

# Run all schemes starting from the exact same master initial array
ftcs_snapshots, ftcs_times = solve_ftcs(x, u0_master, alpha, dt, dx, Nt)
btcs_snapshots, btcs_times = solve_btcs(x, u0_master, alpha, dt, dx, Nt)
cn_snapshots, cn_times = solve_crank_nicolson(x, u0_master, alpha, dt, dx, Nt)
expm_snapshots, expm_times = solve_matrix_exponential(x, u0_master, alpha, dt, dx, Nt)

# Verify that u0_master was not changed during executions
reference_u0 = apply_boundary_conditions(get_initial_condition(x), 0.0)
is_u0_preserved = np.array_equal(u0_master, reference_u0)

print(f"Initial condition array preserved? {is_u0_preserved} (True is correct!)")
print(f"Snapshots computed -> FTCS: {len(ftcs_snapshots)}, BTCS: {len(btcs_snapshots)}, CN: {len(cn_snapshots)}, Expm: {len(expm_snapshots)}")

In [ ]:
# ==============================================================================
# 8. Graphical Comparison of the Results
# ==============================================================================
plt.figure(figsize=(10, 6))

plt.plot(x, u0_master, 'k--', label='Initial Condition (t=0)')
plt.plot(x, ftcs_snapshots[-1], 'ro-', label=f'FTCS (Explicit) at t={ftcs_times[-1]:.3f}s')
plt.plot(x, btcs_snapshots[-1], 'bs-', label=f'BTCS (Implicit) at t={btcs_times[-1]:.3f}s')
plt.plot(x, cn_snapshots[-1], 'g^-', label=f'Crank-Nicolson at t={cn_times[-1]:.3f}s')
plt.plot(x, expm_snapshots[-1], 'y*-', label=f'Matrix Exp (expm_multiply) at t={expm_times[-1]:.3f}s')

plt.xlabel('x (position)')
plt.ylabel('Temperature')
plt.title(f'Comparison of Schemes at t = {t_final:.2f} s')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 9. Dynamic Animation Comparison (2x2 Layout)
# ==============================================================================
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))
axes = [ax1, ax2, ax3, ax4]
names = ['FTCS (Explicit)', 'BTCS (Implicit)', 'Crank-Nicolson', 'Matrix Exponential (expm_multiply)']
colors = ['r', 'b', 'g', 'm']

for ax, name in zip(axes, names):
    ax.set_xlim(0, L)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('x')
    ax.set_ylabel('Temperature')
    ax.set_title(name)
    ax.grid(True)

line_ftcs, = ax1.plot([], [], 'r-', lw=2)
line_btcs, = ax2.plot([], [], 'b-', lw=2)
line_cn, = ax3.plot([], [], 'g-', lw=2)
line_expm, = ax4.plot([], [], 'm-', lw=2)

time_text1 = ax1.text(0.05, 0.9, '', transform=ax1.transAxes)
time_text2 = ax2.text(0.05, 0.9, '', transform=ax2.transAxes)
time_text3 = ax3.text(0.05, 0.9, '', transform=ax3.transAxes)
time_text4 = ax4.text(0.05, 0.9, '', transform=ax4.transAxes)

def init_anim():
    line_ftcs.set_data([], [])
    line_btcs.set_data([], [])
    line_cn.set_data([], [])
    line_expm.set_data([], [])
    time_text1.set_text('')
    time_text2.set_text('')
    time_text3.set_text('')
    time_text4.set_text('')
    return line_ftcs, line_btcs, line_cn, line_expm, time_text1, time_text2, time_text3, time_text4

def update_anim(frame):
    line_ftcs.set_data(x, ftcs_snapshots[frame])
    time_text1.set_text(f't = {ftcs_times[frame]:.3f} s')
    
    line_btcs.set_data(x, btcs_snapshots[frame])
    time_text2.set_text(f't = {btcs_times[frame]:.3f} s')
    
    line_cn.set_data(x, cn_snapshots[frame])
    time_text3.set_text(f't = {cn_times[frame]:.3f} s')
    
    line_expm.set_data(x, expm_snapshots[frame])
    time_text4.set_text(f't = {expm_times[frame]:.3f} s')
    
    return line_ftcs, line_btcs, line_cn, line_expm, time_text1, time_text2, time_text3, time_text4

num_frames = len(ftcs_snapshots)
anim = FuncAnimation(
    fig,
    update_anim,
    frames=num_frames,
    init_func=init_anim,
    interval=200,
    blit=True
)

plt.tight_layout()
plt.show()